### FEATURE ENGINEERING EXPERIMENTATION DECK

In [1]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_parquet("../data/processed/scraper.parquet") 
df.head()

,product_id,product_name,brand,price,product_rating,category,fetched_time,source
0,71ac68772712f55cc12c20e7d7117504,"Blueing 15.6"" Laptop J4125 8GB+256GB SSD Stude...",Blueing,234000.0,3.9,laptops,2025-11-13 01:35:13+00:00,Web_scraper
1,979798712fb36233548099e1a7dc6b5e,Macbook PRO Laptop A1278 13.3 Inch Core I5 2.5...,Macbook,187000.0,3.4,laptops,2025-11-13 01:35:13+00:00,Web_scraper
2,8c89bf1aac7bfa5648cae0754ab0f57f,"Blueing 14"" Laptop N3350 6GB+192GB SSD Portabl...",Blueing,215278.0,4.1,laptops,2025-11-13 01:35:13+00:00,Web_scraper
3,dbb782b45290f360b6cb6c421c0ad2b8,Hp EliteBook 840 G7 Intel Core I5 Touchscreen ...,Hp,566000.0,4.0,laptops,2025-11-13 01:35:13+00:00,Web_scraper
4,c81885a7888ef229d8d34abba9d31c94,Hp Hp Hp EliteBook 840 G7 10th Gen Intel Core ...,Hp,519990.0,4.1,laptops,2025-11-13 01:35:13+00:00,Web_scraper


In [4]:
df_2 = pd.read_parquet("../data/features/api_ingest/product_features.parquet")
df_2.head() 

,product_id,product_name,brand,seller_name,source,first_seen,last_seen,days_active,observation_count,avg_price,current_price,price_tier,avg_rating,current_rating,total_rating_count,stock_qty
0,5000867,JCO4 Laptop Battery – High Capacity – Recharge...,HP,Konga,api_ingest,2025-11-11 19:49:08+00:00,2026-05-26 02:57:16+00:00,195,43,35062.44186,49420,High,0.0,0.0,0.0,1
1,6037430,Prodesk Adjustable Laptop Stand,Nillkin,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2026-02-16 02:01:40+00:00,96,22,60000.00000,60000,High,5.0,5.0,22.0,14
2,6804247,Atheris Wireless Mouse,Razer,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2026-02-16 02:01:40+00:00,96,22,255000.00000,255000,High,0.0,0.0,0.0,20
3,6804244,Naga Pro Wireless Gaming Mouse - Black,Razer,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2026-02-16 02:01:40+00:00,96,22,340000.00000,340000,High,0.0,0.0,0.0,20
4,6815170,2024 Magic Mouse - Multi-touch Surface - Usb-c...,Unknown,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2026-02-16 02:01:40+00:00,96,22,190000.00000,190000,High,0.0,0.0,0.0,2


In [27]:
df.dtypes

product_id                      int64
product_name                      str
brand                             str
price                           int64
product_rating                float64
rating_count                  float64
seller_name                       str
stock_qty                       int64
category                       object
fetched_time      datetime64[us, UTC]
source                            str
dtype: object

In [28]:
df.drop(columns="category", inplace=True)
df.isna().sum()

product_id         0
product_name       0
brand              0
price              0
product_rating    40
rating_count      40
seller_name        0
stock_qty          0
fetched_time       0
source             0
dtype: int64

In [29]:
# Binning different price tiers into low, mid & high
df["price_tier"] = pd.qcut(df["price"], q=3, labels= ["Low", "Mid", "High"]) 

### BUILDING THE PRODUCT INTELLIGENCE ENGINE

In [30]:
# Checking the distribution of product_id inorder to investigate the distribution of products within the dataset
df["product_id"].value_counts()

product_id
5000867    43
6566278    22
6861874    22
6860173    22
6860155    22
           ..
6986469     1
6986465     1
6986464     1
6986463     1
6986457     1
Name: count, Length: 664, dtype: int64

In [32]:
# Sorting the dataset by fetched_time and product_id to ensure that the latest price and rating information is captured for each product 
df = df.sort_values(["fetched_time", "product_id"]) 
df.head()

,product_id,product_name,brand,price,product_rating,rating_count,seller_name,stock_qty,fetched_time,source,price_tier
3,2353031,HP Big-Mouth Laptop Charger - 18.5V 3.5A,HP,10000,4.5,2.0,YOMILINCON BRAND,9,2025-11-11 19:49:08+00:00,api_ingest,Low
0,5000867,JCO4 LAPTOP BATTERY,HP,24725,0.0,0.0,Konga,1,2025-11-11 19:49:08+00:00,api_ingest,Mid
2,5228829,Ht03xl In-built Battery,HP,35075,0.0,0.0,Konga,2,2025-11-11 19:49:08+00:00,api_ingest,Mid
39,5522604,C270 Hd Webcam - Hd 720p - Widescreen Hd Video,Logitech,80000,0.0,0.0,YOMILINCON BRAND,17,2025-11-11 19:49:08+00:00,api_ingest,Mid
38,5527693,Mini Display Port To Hdmi,Mini,36200,0.0,0.0,YOMILINCON BRAND,17,2025-11-11 19:49:08+00:00,api_ingest,Mid


In [33]:
df_2 = df.groupby("product_id", dropna=False).agg(product_name = ("product_name", "last"), brand = ("brand", "last"), seller_name = ("seller_name", "last"), source = ("source", "last"), first_seen = ("fetched_time", "first"), last_seen = ("fetched_time", "last"),
                                                   observation_count = ("product_id", "size"), avg_price = ("price", "mean"), current_price = ("price", "last"), avg_rating = ("product_rating", "mean"), current_rating = ("product_rating", "last"), 
                                                   total_rating_count = ("product_rating", "sum"), stock_qty = ("stock_qty", "last")).reset_index()
df_2.head() 

,product_id,product_name,brand,seller_name,source,first_seen,last_seen,observation_count,avg_price,current_price,avg_rating,current_rating,total_rating_count,stock_qty
0,2353031,HP Big-Mouth Laptop Charger - 18.5V 3.5A,HP,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2025-11-11 19:49:08+00:00,1,10000.00000,10000,4.5,4.5,4.5,9
1,4835380,Flexible Usb External Keyboard And Optical Wir...,Havit,YOMILINCON BRAND,api_ingest,2025-12-21 01:41:53+00:00,2026-01-16 01:40:10+00:00,7,10000.00000,10000,0.0,0.0,0.0,15
2,4840254,Optical Usb Mouse,Dell,KACHISON,api_ingest,2026-03-31 02:11:08+00:00,2026-03-31 02:11:08+00:00,1,4000.00000,4000,5.0,5.0,5.0,1
3,4851063,Rgb Backlit Gaming Mouse -black,Havit,YOMILINCON BRAND,api_ingest,2025-12-16 01:28:12+00:00,2025-12-26 01:27:46+00:00,3,19500.00000,19500,4.0,4.0,12.0,7
4,5000867,JCO4 Laptop Battery – High Capacity – Recharge...,HP,Konga,api_ingest,2025-11-11 19:49:08+00:00,2026-05-26 02:57:16+00:00,43,35062.44186,49420,0.0,0.0,0.0,1


### COMPLETED THE PRODUCT FEATURE ENGINEERING NEEDED FOR PRODUCT INTELLIGENCE! 

In [34]:
df_2["price_tier"] = pd.qcut(df_2["current_price"], q=3, labels= ["Low", "Mid", "High"])  

### EXPERIMENTATION COMPLETED - The brand & seller intelligence entails thesame steps which was taken in the product intelligence, Therefore, all i need to do is repurpose the steps and actions carried out in the product intelligence for the rest!! 

In [35]:
df["brand"].value_counts()

brand
Apple       220
Logitech    141
Razer       110
Hp           72
Baseus       67
           ... 
LDNIO         1
2.4g          1
Lingo         1
Pc            1
Jabra         1
Name: count, Length: 169, dtype: int64

In [36]:
df_2 = df.groupby("brand")
df_2["brand"].value_counts()

brand
/           1
1.5m        1
100w        1
10M         2
10meters    2
           ..
Wireless    1
Wiwu        1
X75v2       1
Xiaomi      1
Yvi+        1
Name: count, Length: 169, dtype: int64

In [1]:
import pandas as pd
from pathlib import Path

base = Path("../data/features/api_ingest")

product = pd.read_parquet(base / "product_features.parquet")
brand = pd.read_parquet(base / "brand_features.parquet")
seller = pd.read_parquet(base / "seller_features.parquet") 

In [3]:
print("PRODUCT FEATURES")
product.head(10) 

PRODUCT FEATURES


,product_id,product_name,brand,seller_name,source,first_seen,last_seen,days_active,observation_count,avg_price,current_price,price_tier,avg_rating,current_rating,total_rating_count,stock_qty
0,5000867,JCO4 Laptop Battery – High Capacity – Recharge...,HP,Konga,api_ingest,2025-11-11 19:49:08+00:00,2026-05-26 02:57:16+00:00,195,43,35062.44186,49420,High,0.0,0.0,0.0,1
1,6037430,Prodesk Adjustable Laptop Stand,Nillkin,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2026-02-16 02:01:40+00:00,96,22,60000.00000,60000,High,5.0,5.0,22.0,14
2,6804247,Atheris Wireless Mouse,Razer,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2026-02-16 02:01:40+00:00,96,22,255000.00000,255000,High,0.0,0.0,0.0,20
3,6804244,Naga Pro Wireless Gaming Mouse - Black,Razer,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2026-02-16 02:01:40+00:00,96,22,340000.00000,340000,High,0.0,0.0,0.0,20
4,6815170,2024 Magic Mouse - Multi-touch Surface - Usb-c...,Unknown,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2026-02-16 02:01:40+00:00,96,22,190000.00000,190000,High,0.0,0.0,0.0,2
5,5528744,45W Magsafe 2 Power Adapter,Apple,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2026-02-16 02:01:40+00:00,96,22,124000.00000,124000,High,0.0,0.0,0.0,20
6,5528748,60W Magsafe Power Adapter,Apple,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2026-02-16 02:01:40+00:00,96,22,127000.00000,127000,High,0.0,0.0,0.0,20
7,6832811,Apple 2024 Magic Mouse - Multi-touch Surface -...,Apple,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2026-02-16 02:01:40+00:00,96,22,260000.00000,260000,High,0.0,0.0,0.0,5
8,6799336,Apple Magic Mouse - Wireless / Bluetooth / Re...,Apple,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2026-02-16 02:01:40+00:00,96,22,245000.00000,245000,High,0.0,0.0,0.0,20
9,6037417,Ascent Stand For Laptops,Nillkin,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2026-02-16 02:01:40+00:00,96,22,17400.00000,17400,Mid,0.0,0.0,0.0,20


In [2]:
print(product.shape)
print(product.info())

(664, 16)
<class 'pandas.DataFrame'>
RangeIndex: 664 entries, 0 to 663
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   product_id          664 non-null    int64              
 1   product_name        664 non-null    str                
 2   brand               664 non-null    str                
 3   seller_name         664 non-null    str                
 4   source              664 non-null    str                
 5   first_seen          664 non-null    datetime64[us, UTC]
 6   last_seen           664 non-null    datetime64[us, UTC]
 7   days_active         664 non-null    int64              
 8   observation_count   664 non-null    int64              
 9   avg_price           664 non-null    float64            
 10  current_price       664 non-null    int64              
 11  price_tier          664 non-null    category           
 12  avg_rating          634 non-null    f

In [5]:
print("BRAND FEATURES")
brand.head(10)

BRAND FEATURES


,brand,product_count,observation_count,seller_count,top_seller,avg_price,median_price,price_tier,avg_rating,total_rating_count,rating_coverage_pct,avg_stock_qty,source
0,Unknown,241,446,57,Add Moore,24711.821577,15000.0,Low,0.042696,35.0,1.244813,16.846473,api_ingest
1,HP,40,124,21,Add Moore,34768.625000,28000.0,Mid,0.132353,2.0,2.500000,16.950000,api_ingest
2,Logitech,26,141,10,YOMILINCON BRAND,345973.000000,82500.0,High,0.051282,3.0,3.846154,7.346154,api_ingest
3,Apple,25,220,14,YOMILINCON BRAND,125620.000000,92000.0,High,0.000000,0.0,0.000000,17.560000,api_ingest
4,Dell,23,23,16,Raymond Gadgets And Accessories,50300.000000,25000.0,Mid,0.238095,1.0,4.347826,14.826087,api_ingest
5,Lenovo,16,28,10,Ambless hub,14081.187500,10750.0,Low,0.306818,13.0,6.250000,17.875000,api_ingest
6,ASUS,15,15,9,More Ventures,11880.000000,8000.0,Low,0.000000,0.0,0.000000,19.333333,api_ingest
7,High,14,14,7,PEAKEXCEL TECHNOLOGIES,20710.642857,12999.5,Low,0.000000,0.0,0.000000,20.000000,api_ingest
8,Microsoft,11,47,5,AKAIT COMMUNICATIONS,210627.272727,168500.0,High,0.000000,0.0,0.000000,6.090909,api_ingest
9,Type-C,11,13,10,Raymond Gadgets And Accessories,11863.636364,9800.0,Low,0.000000,0.0,0.000000,16.545455,api_ingest


In [4]:
print(brand.shape)
print(brand.dtypes)

(127, 13)
brand                       str
product_count             int64
observation_count         int64
seller_count              int64
top_seller                  str
avg_price               float64
median_price            float64
price_tier             category
avg_rating              float64
total_rating_count      float64
rating_coverage_pct     float64
avg_stock_qty           float64
source                      str
dtype: object


In [6]:
print("\nSELLER FEATURES")
print(seller.shape)
print(seller.dtypes)


SELLER FEATURES
(114, 13)
seller_name                 str
product_count             int64
observation_count         int64
brand_count               int64
top_brand                   str
avg_price               float64
median_price            float64
price_tier             category
avg_rating              float64
total_rating_count      float64
rating_coverage_pct     float64
avg_stock_qty           float64
source                      str
dtype: object


In [7]:
seller.head(20)

,seller_name,product_count,observation_count,brand_count,top_brand,avg_price,median_price,price_tier,avg_rating,total_rating_count,rating_coverage_pct,avg_stock_qty,source
0,YOMILINCON BRAND,46,860,16,Apple,165650.465116,130000.0,High,0.161047,30.0,3.372093,15.134884,api_ingest
1,Highspeed,44,44,19,Unknown,20604.545455,13475.0,Low,0.000000,0.0,0.000000,19.954545,api_ingest
2,Add Moore,41,45,14,Unknown,18404.444444,14300.0,Low,0.000000,0.0,0.000000,20.000000,api_ingest
3,Giga ventures,38,38,15,Unknown,18679.947368,12499.5,Low,0.000000,0.0,0.000000,20.000000,api_ingest
4,More Ventures,38,38,17,Unknown,17635.473684,13000.0,Low,0.000000,0.0,0.000000,20.000000,api_ingest
5,Royal Ventures,34,37,14,Unknown,24572.972973,14000.0,Low,0.000000,0.0,0.000000,20.000000,api_ingest
6,GiftedVentures,33,35,17,Unknown,20129.942857,15999.0,Mid,0.000000,0.0,0.000000,20.000000,api_ingest
7,Nachi Ventures,31,35,14,Unknown,26718.285714,16000.0,Mid,0.000000,0.0,0.000000,20.000000,api_ingest
8,Raymond Gadgets And Accessories,24,28,10,Unknown,21607.142857,19000.0,Mid,0.000000,0.0,0.000000,2.000000,api_ingest
9,Maxpower Ventures,18,18,11,Unknown,17613.888889,11250.0,Low,0.000000,0.0,0.000000,20.000000,api_ingest


## TIME TO FOCUS ON THE SCRAPER DATASET AND THE FEATURE ENGINEERING PROCESS !!